# Detectives del Código

Cada ejercicio viola un principio SOLID

En equipo:
1. Lee el código y ejecútalo.
2. Responde: ¿qué principio se viola? 
3. ¿Por qué? : señala qué parte del código lo causa y explica con tus propiar palabras por que.



## Ejercicio 01: Cancelación de Pólizas

`PolizaColectiva` hereda de `Poliza`, pero no permite hacer lo mismo que su clase base.


In [3]:
class Poliza:
    def __init__(self, suma_asegurada):
        self.suma_asegurada = suma_asegurada
        self.activa = True
        
    def cancelar(self):
        self.activa = False
        return "Póliza cancelada"

class PolizaColectiva(Poliza):
    def cancelar(self):
        raise Exception("Las pólizas colectivas no se cancelan individualmente")


def procesar_cancelacion(poliza):
        """Función genérica: no debería importar de qué tipo es la póliza."""
        resultado = poliza.cancelar()
        print(resultado)

#Ejecucion

procesar_cancelacion(Poliza(500000))
try:
    procesar_cancelacion(PolizaColectiva(2000000))
except Exception as e:
    print(f"Error inesperado: {e}")


Póliza cancelada
Error inesperado: Las pólizas colectivas no se cancelan individualmente


In [1]:
from IPython.display import HTML

html = """
<div style="font-family: Helvetica, Arial, sans-serif; display: flex; flex-direction: column; align-items: center; gap: 6px; padding: 20px;">

  <!-- Clase Poliza -->
  <div style="border: 1.5px solid #333; width: 260px; border-radius: 4px; overflow: hidden;">
    <div style="background:#eee; font-weight:bold; text-align:center; padding:6px; border-bottom:1.5px solid #333;">
      Poliza
    </div>
    <div style="padding:6px 10px; border-bottom:1px solid #999; font-size:13px;">
      + suma_asegurada<br>
      + activa
    </div>
    <div style="padding:6px 10px; font-size:13px;">
      + __init__(suma_asegurada)<br>
      + cancelar(): str
    </div>
  </div>

  <!-- Flecha de herencia -->
  <div style="text-align:center; font-size:13px; color:#333;">
    △<br>
    <span style="font-size:12px;">hereda</span><br>
    │
  </div>

  <!-- Clase PolizaColectiva -->
  <div style="border: 2px solid #c0392b; width: 260px; border-radius: 4px; overflow: hidden;">
    <div style="background:#f8d7d7; font-weight:bold; text-align:center; padding:6px; border-bottom:2px solid #c0392b; color:#922; ">
      PolizaColectiva
    </div>
    <div style="padding:6px 10px; border-bottom:1px solid #e0aaaa; font-size:13px; color:#922;">
      &nbsp;
    </div>
    <div style="padding:6px 10px; font-size:13px; color:#922;">
      + cancelar(): str<br>
      &nbsp;&nbsp;raise Exception(...) ⚠
    </div>
  </div>

</div>
"""

display(HTML(html))

Se viola el principio de <b>Sustitución de Liskov</b>

Si S es una subclase de T, entonces los objetos de tipo T deben poder reemplazarse por objetos de tipo S sin alterar el correcto funcionamiento del programa.

Si PolizaColectiva es una subclase de Poliza, entonces los objetos de tipo Poliza deben poder reemplazarse por objetos de tipo PolizaColectiva sin alterar el correcto funcionamiento del programa.

### Algunas Pistas para detectar si se viola este principio:

- El método padre nunca lanzaba error, pero el de la subclase sí. 
- El método sobrescrito se queda a medias, o  no hace nada (pass), cuando el padre sí hacía el trabajo completo.
"Ahora pide más requisitos"
- La clase padre aseguraba cierto resultado (por ejemplo, "el saldo nunca queda negativo"), pero la subclase ya no lo garantiza.


## Solución

In [2]:
class Poliza:
    """Clase base con lo que TODAS las pólizas comparten, sin prometer cancelar()."""
    def __init__(self, suma_asegurada):
        self.suma_asegurada = suma_asegurada
        self.activa = True

    def info(self):
        return f"Póliza por ${self.suma_asegurada}, activa: {self.activa}"


class PolizaIndividual(Poliza):
    """Aquí sí vive la promesa de poder cancelarse uno mismo."""
    def cancelar(self):
        self.activa = False
        return "Póliza cancelada"


class PolizaColectiva(Poliza):
    """No hereda cancelar() porque nunca tuvo ese comportamiento."""
    def cancelar_grupal(self, autorizacion_comite):
        self.activa = False
        return "Póliza colectiva cancelada por autorización del comité"


def procesar_cancelacion(poliza: PolizaIndividual):
    # Esta función solo recibe pólizas que sí saben cancelarse individualmente
    print(poliza.cancelar())


procesar_cancelacion(PolizaIndividual(500000))
# procesar_cancelacion(PolizaColectiva(2000000))  # ni siquiera es válido pasarlo: no tiene cancelar()

Póliza cancelada


## Ejercicio 02: Registro de Siniestros

El registrador de siniestros está atado directamente a una sola forma de guardar la información: un archivo de texto.


In [7]:
class ArchivoTexto:
    def guardar(self, registro):
        with open("siniestros.txt", "a", encoding="utf-8") as f:
            f.write(registro + "\n")


class RegistradorSiniestro:
    def __init__(self, almacen):
        self.almacen = almacen  # atado a una implementación concreta

    def registrar(self, siniestro):
        self.almacen.guardar(siniestro)


# Ejecucion
registrador = RegistradorSiniestro(ArchivoTexto())
registrador.registrar("Siniestro #4521 - Auto - $35,000")

with open("siniestros.txt", encoding="utf-8") as f:
    print("Contenido del archivo:")
    print(f.read())


Contenido del archivo:
Siniestro #4521 - Auto - $35,000
Siniestro #4521 - Auto - $35,000
Siniestro #4521 - Auto - $35,000



¿Qué modificaciones se harían si ahora la aseguradora desea guardad los siniestros en un archivo CSV, luego en una BD?


<details>
<summary><b>Ver respuesta</b></summary>

**Principio violado: D (Inversión de Dependencias)**

`RegistradorSiniestro` crea directamente un `ArchivoTexto` dentro de su propio constructor. Si cambia la forma de guardar los siniestros (CSV, Excel, otro archivo), hay que modificar `RegistradorSiniestro`, aunque su trabajo (registrar el siniestro) no cambió en absoluto.

**Corrección:** recibir el almacén como parámetro (inyección de dependencia), en vez de crearlo adentro.
</details>

## Ejercicio 03: Emisión de Póliza

Una sola clase calcula la prima, genera el documento de la póliza y notifica al asegurado.


In [4]:
class Poliza:
    def __init__(self, suma_asegurada, tasa):
        self.suma_asegurada = suma_asegurada
        self.tasa = tasa

    def calcular_prima(self):
        return self.suma_asegurada * self.tasa

    def generar_documento_pdf(self):
        print(f"Generando carátula de póliza. Prima anual: ${self.calcular_prima():.2f}")

    def notificar_asegurado(self, email):
        print(f"Enviando póliza a {email}")


poliza = Poliza(suma_asegurada=500000, tasa=0.012)
poliza.generar_documento_pdf()
poliza.notificar_asegurado("asegurado@correo.com")


Generando carátula de póliza. Prima anual: $6000.00
Enviando póliza a asegurado@correo.com


## Ejercicio 04: Reserva y Beneficio por Fallecimiento

Una sola clase base obliga a todos los productos a tener los mismos métodos, aunque no les apliquen.


In [5]:
class ProductoActuarial:
    def calcular_reserva(self):
        raise NotImplementedError

    def pagar_beneficio_fallecimiento(self):
        raise NotImplementedError


class SeguroVida(ProductoActuarial):
    def calcular_reserva(self):
        return 45000.0

    def pagar_beneficio_fallecimiento(self):
        return 500000.0


class SeguroAuto(ProductoActuarial):
    def calcular_reserva(self):
        return 8000.0

    def pagar_beneficio_fallecimiento(self):
        return 0.0  # un seguro de auto no paga beneficio por fallecimiento


# Demo
productos = [SeguroVida(), SeguroAuto()]
for p in productos:
    print(type(p).__name__, "reserva:", p.calcular_reserva(),
          "| beneficio fallecimiento:", p.pagar_beneficio_fallecimiento())


SeguroVida reserva: 45000.0 | beneficio fallecimiento: 500000.0
SeguroAuto reserva: 8000.0 | beneficio fallecimiento: 0.0


## Ejercicio 05: Prima según Tipo de Seguro

Una función calcula la prima según el tipo de seguro, usando `if/elif`.


In [6]:
def calcular_prima(tipo_seguro, suma_asegurada):
    if tipo_seguro == "vida":
        return suma_asegurada * 0.012
    elif tipo_seguro == "auto":
        return suma_asegurada * 0.035
    else:
        return suma_asegurada * 0.012  # por defecto, se cotiza como vida


# Demo
print("Vida:", calcular_prima("vida", 500000))
print("Auto:", calcular_prima("auto", 200000))
print("Gastos médicos (no existe todavía):", calcular_prima("gastos_medicos", 300000))  # se cotiza mal, sin avisar


Vida: 6000.0
Auto: 7000.000000000001
Gastos médicos (no existe todavía): 3600.0
